In [133]:
from backtesting import Backtest, Strategy
import pandas as pd
import math
from datetime import time
import yfinance as yf

In [134]:
Ticker = "NQ=F"

df = yf.download(Ticker, period="max", interval="15m")

df.columns.names = [None, None]

df.columns = df.columns.get_level_values(0)

df = df.drop(columns=["Volume"])

df

[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open
Datetime,,,,
2026-06-16 18:00:00-04:00,30363.50,30375.00,30306.50,30306.50
2026-06-16 18:15:00-04:00,30366.75,30377.00,30343.75,30363.75
2026-06-16 18:30:00-04:00,30364.25,30381.25,30331.25,30368.00
2026-06-16 18:45:00-04:00,30378.00,30389.25,30353.00,30364.75
2026-06-16 19:00:00-04:00,30350.25,30380.00,30326.00,30378.75
...,...,...,...,...
2026-08-14 15:45:00-04:00,30145.00,30148.00,30101.50,30108.50
2026-08-14 16:00:00-04:00,30132.75,30153.25,30131.00,30144.25
2026-08-14 16:15:00-04:00,30131.00,30135.50,30127.25,30133.25


In [135]:
# df = pd.read_csv("xauusd-m1-bid-2020-01-01-2026-08-15.csv")

# df.set_index("Datetime", inplace=True)

# df = df[["Open", "High", "Low", "Close"]]

# df.index = pd.to_datetime(df.index, unit="ms")

# df = df.tail(50000)

# df

In [136]:
def get_digits(price_step):
    return round(-math.log10(price_step))

In [137]:
BALANCE         = 100_000

RRR = 2

CONTRACT_SIZE   = 10
LOT_SIZE        = 0.05

RISK = 1000 # $

RANGE_LENGTH    = 1
RANGE_TIME      = "09:30"
END_OF_DAY_TIME = "16:00"

GAMMA = 2

# e.g. EURUSD=X has a price step of 0.0001
# e.g. XAUUSD has a price step of 0.01
TICKER_PRICE_PRECISION = get_digits(0.01)

In [138]:
class ORB(Strategy):
    def init(self): 
        self.long_order = None
        self.short_order = None

    def next(self):
        _current_time = self.data.index[-1].strftime("%H:%M")

        if _current_time == RANGE_TIME:
            _range_high = max(self.data.High[-RANGE_LENGTH:])
            _range_low = min(self.data.Low[-RANGE_LENGTH:])

            _delta = (_range_high - _range_low) * GAMMA

            if _delta > 0:
                # "size must be a positive fraction of equity, or a positive whole number of units"
                _units = round(RISK / _delta)

                self.long_order = self.buy(
                    size=_units,
                    stop=_range_high,
                    sl=_range_high - _delta,
                    tp= _range_high + RRR * _delta
                )

                self.short_order = self.sell(
                    size=_units,
                    stop=_range_low,
                    sl=_range_high + _delta,
                    tp= _range_low - RRR * _delta
                )

        if self.position.is_long:
            if self.short_order is not None:
                try: self.short_order.cancel()
                except ValueError: pass
                finally: self.short_order = None

        elif self.position.is_short:
            if self.long_order is not None:
                try: self.long_order.cancel()
                except ValueError: pass
                finally: self.long_order = None

        if _current_time == END_OF_DAY_TIME:
            # Close open positions
            for trade in self.trades: trade.close()

            # Cancel pending orders
            for order in self.orders: order.cancel()

        return


# ============================================================
# Backtest
# ============================================================

bt = Backtest(
    df,
    ORB,
    cash=BALANCE,
    margin=1/1000,
    commission=0.0,
    exclusive_orders=False,
    hedging=True,
    finalize_trades=True
)

In [139]:
stats = bt.run()

print(stats)

Start                     2026-06-16 18:00...
End                       2026-08-14 16:45...
Duration                     58 days 22:45:00
Exposure Time [%]                    98.35079
Equity Final [$]                      42854.5
Equity Peak [$]                     242367.75
Return [%]                           -57.1455
Buy & Hold Return [%]                -0.73032
Return (Ann.) [%]                   -98.71941
Volatility (Ann.) [%]                27.71305
CAGR [%]                            -99.47544
Sharpe Ratio                          -3.5622
Sortino Ratio                        -0.53573
Calmar Ratio                         -1.14013
Alpha [%]                           -63.70649
Beta                                 -8.98375
Max. Drawdown [%]                   -86.58609
Avg. Drawdown [%]                    -8.39444
Max. Drawdown Duration       16 days 00:30:00
Avg. Drawdown Duration        1 days 12:25:00
# Trades                                   30
Win Rate [%]                      

In [140]:
# bt.plot()